## Final Project Submission

Please fill out:
* Student name: 
* Student pace: self paced / part time / full time
* Scheduled project review date/time: 
* Instructor name: 
* Blog post URL:


## **Business Understanding**

### ***The Business Problem***
Your company now sees all the big companies creating original video content and they want to get in on the fun. They have decided to create a new movie studio, but they don’t know anything about creating movies. You are charged with exploring what types of films are currently doing the best at the box office. You must then translate those findings into actionable insights that the head of your company's new movie studio can use to help decide what type of films to create.

### ***Key Stakeholders***
+ Studio Management
     + Studio Manager
     + Operatios Managers
     + Finance Manager (CFO)
+ Producers and studio crew

### ***Key Objectives/Business questions***
1. Which genres deliver the best financial return? (measured by ROI and median profit) 
2. Which genres do best with an international audience 
3. Which studios perform best and do they align with the top genres

### *Key Metrics of Success according to studio KPIs*


---

## **Data Understanding**
+ For this analysis, I will at the bom.movies_gross.csv.gz dataset for analysis using pandas.
+ For SQL, the database to be used is the im.db, specifically the movie_basics and movie_ratings tables

The main objectives in this initial data understanding is to:
+ Find the column names
+ The shape
+ Any missing values and duplicates
+ Understand the data types 
+ etc

In [6]:
# import the needed libraries and modules:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sqlite3

In [7]:
# Make a connection to the database for the sqlite
conn = sqlite3.connect('zippedData/im.db/im.db')

In [8]:
cur = conn.cursor()

In [9]:
bommovies_df = pd.read_csv("zippedData/bom.movie_gross.csv.gz")

#### **bom.movies_gross.csv.gz**

In [10]:
bommovies_df.head()

,title,studio,domestic_gross,foreign_gross,year
0,Toy Story 3,BV,415000000.0,652000000,2010
1,Alice in Wonderland (2010),BV,334200000.0,691300000,2010
2,Harry Potter and the Deathly Hallows Part 1,WB,296000000.0,664300000,2010
3,Inception,WB,292600000.0,535700000,2010
4,Shrek Forever After,P/DW,238700000.0,513900000,2010


In [11]:
bommovies_df.shape

# From the ouput below, there are 3,387 rows and 5 columns in the dataset of bom.movie_grosss.csv.gz

(3387, 5)

In [12]:
bommovies_df.tail()

,title,studio,domestic_gross,foreign_gross,year
3382,The Quake,Magn.,6200.0,NaN,2018
3383,Edward II (2018 re-release),FM,4800.0,NaN,2018
3384,El Pacto,Sony,2500.0,NaN,2018
3385,The Swan,Synergetic,2400.0,NaN,2018
3386,An Actor Prepares,Grav.,1700.0,NaN,2018


In [13]:
bommovies_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3387 entries, 0 to 3386
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   title           3387 non-null   object 
 1   studio          3382 non-null   object 
 2   domestic_gross  3359 non-null   float64
 3   foreign_gross   2037 non-null   object 
 4   year            3387 non-null   int64  
dtypes: float64(1), int64(1), object(3)
memory usage: 132.4+ KB


In [14]:
bommovies_df.describe().T

,count,mean,std,min,25%,50%,75%,max
domestic_gross,3359.0,2.874585e+07,6.698250e+07,100.0,120000.0,1400000.0,27900000.0,936700000.0
year,3387.0,2.013958e+03,2.478141e+00,2010.0,2012.0,2014.0,2016.0,2018.0


In [15]:
bommovies_df.dtypes

title              object
studio             object
domestic_gross    float64
foreign_gross      object
year                int64
dtype: object

In [16]:
bommovies_df.duplicated()

0       False
1       False
2       False
3       False
4       False
        ...  
3382    False
3383    False
3384    False
3385    False
3386    False
Length: 3387, dtype: bool

In [17]:
bommovies_df.duplicated().sum()

0

**There are no duplicates**

In [18]:
bommovies_df.isna()

,title,studio,domestic_gross,foreign_gross,year
0,False,False,False,False,False
1,False,False,False,False,False
2,False,False,False,False,False
3,False,False,False,False,False
4,False,False,False,False,False
...,...,...,...,...,...
3382,False,False,False,True,False
3383,False,False,False,True,False
3384,False,False,False,True,False
3385,False,False,False,True,False


In [19]:
bommovies_df.isna().sum()

title                0
studio               5
domestic_gross      28
foreign_gross     1350
year                 0
dtype: int64

In [20]:
# Missing values in %
def missing(bommovies_df,col):
    missing = (bommovies_df[col].isna().sum()/len(bommovies_df))*100
    if missing == 0:
        print(f" we have {missing} % missing from {col}")
    elif missing < 5:
        print(f" we have {missing} % missing from {col} we can go ahead and either drop the rows or fill in the missing values")
    elif missing < 30:
        print(f" we have {missing} % missing from {col} we can go ahead and fill in the missing values")
    else :
        print(f" we have {missing} % missing from {col} we can go ahead and drop the column or consider other alternatives")

In [21]:
missing(bommovies_df,'studio')

 we have 0.14762326542663123 % missing from studio we can go ahead and either drop the rows or fill in the missing values


In [22]:
missing(bommovies_df,'domestic_gross')

 we have 0.8266902863891349 % missing from domestic_gross we can go ahead and either drop the rows or fill in the missing values


In [23]:
missing(bommovies_df, 'foreign_gross')

 we have 39.85828166519043 % missing from foreign_gross we can go ahead and drop the column or consider other alternatives


- studio has missing data of only 5 with a percentage of 0.14. This is insignifanct to warrant dropping the column.
- domestic gross has missing data of 28 with a percentage of 0.83. This is not sufficient enough to have this data dropped.
- foreign gross has missing data. This column has the highest number of NaN the 1350 accounting for the percentage of 39.86. This will not be dropped nevertheless!

**NOTE**
- foreign gross should be float not object 
- year data to be datetime 

### **Movie Data ERD**
![alt text](movie_data_erd.jpeg)


#### **im.db dataset with an a keen interest in the movie_basics and movie_ratings tables**

In [24]:
queryone = pd.read_sql(""" SELECT name FROM sqlite_master WHERE type = 'table';""",conn)
queryone

,name
0,movie_basics
1,directors
2,known_for
3,movie_akas
4,movie_ratings
5,persons
6,principals
7,writers


In [33]:
# im.db movie_ratings

ratings = pd.read_sql("""
SELECT * FROM movie_ratings
;
""", conn)

ratings.head()

,movie_id,averagerating,numvotes
0,tt10356526,8.3,31
1,tt10384606,8.9,559
2,tt1042974,6.4,20
3,tt1043726,4.2,50352
4,tt1060240,6.5,21


In [34]:
ratings.tail()

,movie_id,averagerating,numvotes
73851,tt9805820,8.1,25
73852,tt9844256,7.5,24
73853,tt9851050,4.7,14
73854,tt9886934,7.0,5
73855,tt9894098,6.3,128


In [36]:
ratings.shape

(73856, 3)

In [37]:
ratings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 73856 entries, 0 to 73855
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   movie_id       73856 non-null  object 
 1   averagerating  73856 non-null  float64
 2   numvotes       73856 non-null  int64  
dtypes: float64(1), int64(1), object(1)
memory usage: 1.7+ MB


In [38]:
ratings.describe().T

,count,mean,std,min,25%,50%,75%,max
averagerating,73856.0,6.332729,1.474978,1.0,5.5,6.5,7.4,10.0
numvotes,73856.0,3523.662167,30294.022971,5.0,14.0,49.0,282.0,1841066.0


In [30]:
# im.db movie_basics
basics_querry = pd.read_sql(""" 
SELECT * FROM movie_basics;""",conn)
basics_querry.head()

,movie_id,primary_title,original_title,start_year,runtime_minutes,genres
0,tt0063540,Sunghursh,Sunghursh,2013,175.0,"Action,Crime,Drama"
1,tt0066787,One Day Before the Rainy Season,Ashad Ka Ek Din,2019,114.0,"Biography,Drama"
2,tt0069049,The Other Side of the Wind,The Other Side of the Wind,2018,122.0,Drama
3,tt0069204,Sabse Bada Sukh,Sabse Bada Sukh,2018,NaN,"Comedy,Drama"
4,tt0100275,The Wandering Soap Opera,La Telenovela Errante,2017,80.0,"Comedy,Drama,Fantasy"


In [39]:
basics_querry.tail()

,movie_id,primary_title,original_title,start_year,runtime_minutes,genres
146139,tt9916538,Kuambil Lagi Hatiku,Kuambil Lagi Hatiku,2019,123.0,Drama
146140,tt9916622,Rodolpho Teóphilo - O Legado de um Pioneiro,Rodolpho Teóphilo - O Legado de um Pioneiro,2015,NaN,Documentary
146141,tt9916706,Dankyavar Danka,Dankyavar Danka,2013,NaN,Comedy
146142,tt9916730,6 Gunn,6 Gunn,2017,116.0,None
146143,tt9916754,Chico Albuquerque - Revelações,Chico Albuquerque - Revelações,2013,NaN,Documentary


In [40]:
basics_querry.shape

(146144, 6)

In [41]:
basics_querry.describe().T

,count,mean,std,min,25%,50%,75%,max
start_year,146144.0,2014.621798,2.733583,2010.0,2012.0,2015.0,2017.0,2115.0
runtime_minutes,114405.0,86.187247,166.360590,1.0,70.0,87.0,99.0,51420.0


In [44]:
basics_querry.duplicated().sum()

0

In [46]:
basics_querry.isna().sum()

movie_id               0
primary_title          0
original_title        21
start_year             0
runtime_minutes    31739
genres              5408
dtype: int64

In [50]:
genre = pd.read_sql(""" SELECT genres FROM movie_basics;""",conn)
genre.head()

,genres
0,"Action,Crime,Drama"
1,"Biography,Drama"
2,Drama
3,"Comedy,Drama"
4,"Comedy,Drama,Fantasy"


In [52]:
genre1 = pd.read_sql(""" SELECT original_title, genres FROM movie_basics;""",conn)
genre1.head()

,original_title,genres
0,Sunghursh,"Action,Crime,Drama"
1,Ashad Ka Ek Din,"Biography,Drama"
2,The Other Side of the Wind,Drama
3,Sabse Bada Sukh,"Comedy,Drama"
4,La Telenovela Errante,"Comedy,Drama,Fantasy"


In [26]:
# Your code here - remember to use markdown cells for comments as well!

possible Qs:
- does ratings affect box office success 
- which genre performs well financially - will need to feauture engineer a primary genre column 
- Which studios are most successful, and does that align with the top-performing genres? 
- Out of the films that has a foreign relase, which genre was most successfull amongst international viewers





---
